In [1]:
import os
import pytest
import responses
import requests
from openai import OpenAI
from pydantic import BaseModel, Field, field_validator

# 🔑 Set your API key if it's not already in your system environment variables

os.environ["OPENAI_API_KEY"] = "<Enter your APIs>"

# =====================================================================
# 1. OUTPUT VALIDATION LAYER (Our Pydantic Structural Contract)
# =====================================================================
class OrderLookupSchema(BaseModel):
    order_id: str = Field(description="The alphanumeric order identification string, e.g., ORD-12345.")
    item_quantity: int = Field(description="The quantity of items being queried. Must be a positive integer.")

    # INPUT VALIDATION LAYER inside the Schema (Constraint Testing)
    @field_validator('item_quantity')
    @classmethod
    def verify_positive_quantity(cls, value: int) -> int:
        if value <= 0:
            raise ValueError("Item quantity must be strictly greater than zero.")
        return value

# =====================================================================
# 2. THE API BRIDGE INFRASTRUCTURE
# =====================================================================
def execute_backend_order_search(parsed_data: OrderLookupSchema):
    """
    Simulates sending the AI's validated output across the bridge 
    to our internal warehouse microservice API.
    """
    # Safe string formatting for the URL endpoint
    backend_endpoint = f"https://api.warehouse.internal/v1/orders/{requests.utils.quote(parsed_data.order_id)}"
    params = {"quantity": parsed_data.item_quantity}
    
    response = requests.get(backend_endpoint, params=params, timeout=5)
    return response

print("✅ Cell 1 Executed: Structural contract and secure API bridge compiled successfully.")

✅ Cell 1 Executed: Structural contract and secure API bridge compiled successfully.


In [2]:
@responses.activate
def test_successful_pipeline_flow():
    """
    Ensures that when given clean input, the AI extracts data perfectly, 
    Pydantic approves it, and the backend handles a successful 200 OK.
    """
    # Arrange: Mock out a successful warehouse database response
    responses.add(
        responses.GET,
        "https://api.warehouse.internal/v1/orders/ORD-4412",
        json={"status": "IN_TRANSIT", "delivery_date": "2026-06-15"},
        status=200
    )
    
    client = OpenAI()
    user_input = "Can you look up the status for order number ORD-4412? I bought 3 items."
    
    # Act: AI transforms unstructured text into our strict Pydantic Model
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": user_input}],
        response_format=OrderLookupSchema
    )
    validated_ai_output = completion.choices[0].message.parsed
    
    # Act Part 2: Cross the bridge to our backend API
    api_response = execute_backend_order_search(validated_ai_output)
    
    # Assert
    assert validated_ai_output.order_id == "ORD-4412"
    assert validated_ai_output.item_quantity == 3
    assert api_response.status_code == 200
    assert api_response.json()["status"] == "IN_TRANSIT"
    print("\n[PASSED] Test 1: Positive pipeline & output validation enforced successfully.")

# Run Test 1
pytest.main(["-v", "-s", "-k", "test_successful_pipeline_flow"])

============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /Users/amritansh/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/amritansh/Documents/EY_AI_Test/D3_EY
plugins: anyio-4.13.0
collecting ... collected 0 items

=============================== warnings summary ===============================
../../../anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290
  /Users/amritansh/anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning in 0.00s ==============================


<ExitCode.NO_TESTS_COLLECTED: 5>

In [3]:
def test_input_constraint_failure():
    """
    Verifies that if the user provides an invalid logic constraint (quantity = 0),
    our Pydantic layer catches it and blocks execution before hitting the API.
    """
    # Arrange: Simulate an LLM attempting to pass an invalid zero value
    malicious_extracted_data = {
        "order_id": "ORD-9999",
        "item_quantity": 0  # Violates our custom field validator constraint!
    }
    
    # Act & Assert: Verify that Pydantic raises a ValidationError cleanly
    with pytest.raises(Exception) as exc_info:
        OrderLookupSchema(**malicious_extracted_data)
        
    assert "Item quantity must be strictly greater than zero" in str(exc_info.value)
    print("\n[PASSED] Test 2: Input constraint validation blocked bad metrics before execution.")

# Run Test 2
pytest.main(["-v", "-s", "-k", "test_input_constraint_failure"])

============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /Users/amritansh/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/amritansh/Documents/EY_AI_Test/D3_EY
plugins: anyio-4.13.0
collecting ... collected 0 items

=============================== warnings summary ===============================
../../../anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290
  /Users/amritansh/anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning in 0.00s ==============================


<ExitCode.NO_TESTS_COLLECTED: 5>

In [4]:
@responses.activate
def test_malicious_input_sanitization():
    """
    Tests our defense against hacker scripts trying to hijack our database query strings.
    """
    # Arrange: Mock the backend to handle the escaped string as literal, harmless text
    hostile_id = "ORD-0000'; DROP TABLE Orders; --"
    
    responses.add(
        responses.GET,
        f"https://api.warehouse.internal/v1/orders/{requests.utils.quote(hostile_id)}",
        json={"status": "NOT_FOUND"},
        status=404
    )
    
    # Act: Simulate parsing an injection attack vector
    attack_payload = OrderLookupSchema(order_id=hostile_id, item_quantity=1)
    api_response = execute_backend_order_search(attack_payload)
    
    # Assert: Prove that the string did not break code; it was passed as safe literal text parameter
    assert api_response.status_code == 404
    print("\n[PASSED] Test 3: Malicious SQL injection safely neutralized and contained as text.")

# Run Test 3
pytest.main(["-v", "-s", "-k", "test_malicious_input_sanitization"])

============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /Users/amritansh/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/amritansh/Documents/EY_AI_Test/D3_EY
plugins: anyio-4.13.0
collecting ... collected 0 items

=============================== warnings summary ===============================
../../../anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290
  /Users/amritansh/anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning in 0.00s ==============================


<ExitCode.NO_TESTS_COLLECTED: 5>

In [5]:
@responses.activate
def test_error_handling_validation_500():
    """
    Tests our system's resiliency when the internal warehouse microservice crashes with a 500.
    """
    # Arrange: Force the mock server to simulate a complete database collapse
    responses.add(
        responses.GET,
        "https://api.warehouse.internal/v1/orders/ORD-7777",
        body="Internal Server Database Error",
        status=500
    )
    
    # Act: Create valid arguments and run the execution bridge
    valid_payload = OrderLookupSchema(order_id="ORD-7777", item_quantity=1)
    api_response = execute_backend_order_search(valid_payload)
    
    # Assert: Verify that our bridge catches the 500 status code cleanly so we can alert the AI
    assert api_response.status_code == 500
    print("\n[PASSED] Test 4: Error handling validation caught a backend 500 server crash safely.")

print("📋 Running Full System Verification Test Suite...")
pytest.main(["-v", "-s"])

📋 Running Full System Verification Test Suite...
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /Users/amritansh/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/amritansh/Documents/EY_AI_Test/D3_EY
plugins: anyio-4.13.0
collecting ... collected 0 items

=============================== warnings summary ===============================
../../../anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290
  /Users/amritansh/anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning in 0.00s ==============================


<ExitCode.NO_TESTS_COLLECTED: 5>